# Track 6 example — Which laps were not normal?
### EP0418 · Free-form starter · Formula 1 lap times

Every lap of every race, for every driver, with its time. This is the Jolpica
API, a volunteer-run, free replacement for the old Ergast service, with no key.
Be gentle with it: a few requests a second, a few hundred an hour.

Lap times are a time series with a known shape: a slow first lap, a steady run,
then sudden jumps for pit stops, safety cars and traffic. That makes them a good
test for the anomaly detectors in Lab 4: the "anomalies" have a public explanation
you can check.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, json, time, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

SEASON, ROUND = 2025, 1                   # Australian Grand Prix, Melbourne
CSV = f"f1_{SEASON}_r{ROUND}_laps.csv"


def jolpica(path, limit=100):
    rows, offset = [], 0
    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{path}.json?limit={limit}&offset={offset}"
        with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "EP0418"}), timeout=60) as r:
            d = json.load(r)["MRData"]
        races = d["RaceTable"]["Races"]
        for race in races:
            for lap in race.get("Laps", []):
                for t in lap["Timings"]:
                    rows.append({"race": race["raceName"], "lap": int(lap["number"]),
                                 "driver": t["driverId"], "position": int(t["position"]), "time": t["time"]})
        offset += limit
        if offset >= int(d["total"]):
            break
        time.sleep(0.5)
    return pd.DataFrame(rows)


if not os.path.exists(CSV):
    laps = jolpica(f"{SEASON}/{ROUND}/laps")
    laps.to_csv(CSV, index=False)
laps = pd.read_csv(CSV)
m, s = laps.time.str.split(":", expand=True).iloc[:, 0].astype(int), laps.time.str.split(":", expand=True).iloc[:, 1].astype(float)
laps["seconds"] = m * 60 + s
print(f"{laps.race.iloc[0]}: {laps.lap.max()} laps, {laps.driver.nunique()} drivers, {len(laps)} lap times")
print(laps.groupby("driver").seconds.median().sort_values().head(5).round(3))

### One driver, lap by lap, and the simplest baseline

The baseline for "what will the next lap time be" is **the previous lap**. The
baseline for "is this lap unusual" is a rolling median of the last five laps.
Both are one line, and both are hard to beat for most of the race.

In [ ]:
DRIVER = laps.groupby("driver").seconds.median().idxmin()          # the race's fastest driver by median lap
d = laps[laps.driver == DRIVER].set_index("lap").seconds.sort_index()
baseline = d.shift(1)
mae = (d - baseline).abs().dropna().mean()
print(f"{DRIVER}: 'previous lap' MAE {mae:.3f} s over {d.notna().sum() - 1} laps")

roll = d.rolling(5, min_periods=3).median().shift(1)
flag = (d - roll) > 3.0                                               # three seconds slower than recent laps
print(f"laps more than 3 s slower than the recent median: {list(d.index[flag.fillna(False)])}")
print(">>> Check those lap numbers against the race report: pit stops, safety car, traffic?")

fig, ax = plt.subplots(figsize=(12, 3.5))
d.plot(ax=ax, marker=".", label=DRIVER); roll.plot(ax=ax, ls="--", label="rolling median (last 5)")
ax.scatter(d.index[flag.fillna(False)], d[flag.fillna(False)], c="red", zorder=3, label="flagged")
ax.set_ylabel("lap time (s)"); ax.legend(); ax.set_title(laps.race.iloc[0]); plt.tight_layout(); plt.show()

---
## Three macro questions to shape a proposal

**1. Which anomalies are events and which are noise?**
A pit stop is an anomaly with a cause; a slow lap in traffic is normal racing.
Your detector from Lab 4 will flag both. What extra information separates them,
and is it available during the race or only after? (SLO 5.1, 5.3)

**2. What decision could a lap-time model drive?**
A strategist deciding when to pit; a broadcaster deciding which battle to show; a
fan app predicting the finishing order. Pick one, state its horizon, and say
what it costs when the model is wrong (SLO 7.2).

**3. What would "last year's lap time at this circuit" tell you?**
Jolpica has every season. A baseline that uses the same circuit a year earlier
is a seasonal-naive forecast with a season of one year. When would it beat
"previous lap", and when would it be badly wrong? (SLO 6.1, 6.2)